# Week 1 Walkthrough — Word Embeddings for HN Score Prediction

This notebook narrates the same four-stage pipeline the README's Usage section
describes, calling the repo's *real* modules in sequence rather than reimplementing
any of their logic. The repo is now flat — no nested package directory, no
every import below is a plain `word_embeddings...`, `hn_predictor...`, or a root-level
`config` / `predict` / `serve`.

The four stages, each the same code the matching `make` target runs:

1. **Corpus** — `word_embeddings/etl.py` (`build_corpus()`), run by `make train-corpus`.
   It downloads text8 and queries the HN Postgres mirror, both network-dependent, so
   this notebook does **not** run it. It demonstrates `TextProcessor.normalize_text`
   directly instead, which is the one piece of that stage that is cheap and offline.
2. **Embeddings** — `word_embeddings/train.py` (`train_cbow()`), run by `make train-cbow`.
   This notebook runs it for real with `sample=True`, a tiny built-in corpus and a small
   config, so it finishes in seconds and stays offline (`hf_repo_id=None`,
   `wandb_project=None`).
3. **Predictor** — `hn_predictor/train.py` (`train_predictor()`), run by
   `make train-predictor`. A real run needs the frozen CBOW checkpoint and the 782
   labelled posts, neither of which a quick sample run provides, so this notebook
   narrates the stage and exercises its data loaders rather than training it.
4. **Serving** — `serve.py` (the FastAPI `app`) and `predict.py` (`load_models` /
   `predict_score`). Checkpoints are gitignored and live on the HF Hub, so on a host
   that has not trained or pulled them, `load_models()` raises `FileNotFoundError`.
   That is the designed behaviour, and it is shown rather than hidden.

Run this from the repo root with `uv run jupyter notebook notebooks/walkthrough.ipynb`,
or headlessly with
`uv run jupyter nbconvert --to notebook --execute --inplace notebooks/walkthrough.ipynb`.
Every path in `Config` is repo-root-relative, matching how the `make` targets run.

## 1. Corpus construction (`make train-corpus`)

`word_embeddings/etl.py` builds the merged corpus in one `build_corpus()` call: a
`WikiPipeline` over text8, an `HNPipeline` over the HN mirror, then a vocabulary
merge. Both sources are network-dependent — text8 is downloaded on first use and the
HN half queries a Postgres mirror that is not reachable from every network (the
pipeline falls back to a cleaned extract from the Hub when it cannot connect).

I don't run the full stage here. Instead this cell imports the same module and
exercises `TextProcessor.normalize_text`, the character-replacement table that turns
currencies, dashes and math symbols into ASCII before any vocabulary work happens.
This piece was once dead code (an earlier ASCII strip ran first and deleted every
character the table matches); it is cheap to show it firing for real. Note that
importing `etl` triggers a quiet NLTK data check on first import.

In [ ]:
from word_embeddings.etl import TextProcessor

# build_corpus() (`make train-corpus`) is the real, network-dependent entry point.
# Its text normalization is pure and cheap, so demonstrate that piece directly.
processor = TextProcessor()
for text in [
    "cost: €20 – £15",
    "5 × 3 and a ≤ b",
    "hello 漢 world",
]:
    print(f"{text!r:35} -> {processor.normalize_text(text)!r}")


## 2. Embedding pretraining (`make train-cbow`)

`word_embeddings/train.py`'s `train_cbow(config, sample=False)` is the real
CBOW-with-negative-sampling loop: it builds the dataset, trains `CBOWNS`, evaluates
intrinsically every `cbow_eval_every` batches, and saves the best checkpoint. The
vocabulary travels *inside* the checkpoint alongside the weights, so the embeddings
are never separated from the exact mapping they were indexed by.

Below I call it with `sample=True`, which swaps the real corpus for a tiny built-in
one (`sample_vocabulary()`) and is the same path `make train-cbow-sample` runs. I
point the checkpoint at `cbow_model_sample.pt` (as the `--sample` entry point does)
so this smoke run can never overwrite real embeddings, and I set
`hf_repo_id=None` / `wandb_project=None` so nothing is pushed or tracked.
The real, multi-hour run is `make train-cbow`.

In [ ]:
from config import Config
from word_embeddings.train import train_cbow

config = Config(
    cbow_checkpoint=Config().cbow_checkpoint_dir / "cbow_model_sample.pt",
    cbow_embed_size=32,
    cbow_window_size=2,
    cbow_num_negatives=4,
    cbow_batch_size=8,
    cbow_num_epochs=1,
    hf_repo_id=None,     # never push this throwaway run
    wandb_project=None,  # no tracking either
)

best_loss = train_cbow(config, sample=True)
print(f"Best CBOW loss on the sample corpus: {best_loss:.4f}")
print(f"Checkpoint written to {config.cbow_checkpoint} (gitignored)")


## 3. Predictor training (`make train-predictor`)

`hn_predictor/train.py`'s `train_predictor(config)` freezes the CBOW vectors, averages
each post title into one vector, and trains a small MLP (`SimplePredictor`) to
predict its score. Early stopping matters here: with only 782 labelled posts the
model overfits around epoch 5, so the loop tracks a baseline MSE and stops on
patience. This is the honest stage of the pipeline — the trained model turns out not
to clearly beat predicting the mean.

A real run needs the frozen CBOW checkpoint and the labelled posts, both gitignored
and resolved from the Hub at train time, so I don't train it here. Its data path is
self-contained, though: I exercise `create_title_embeddings` and `SimpleHNDataset`
against a tiny synthetic set of posts and random vectors, so the averaging and
dataset logic run for real without any network or checkpoint.

In [ ]:
import json
import tempfile
from pathlib import Path

import torch

from hn_predictor.data import SimpleHNDataset, create_title_embeddings
from hn_predictor.model import SimplePredictor
from word_embeddings.data import sample_vocabulary

word_to_index, _, _ = sample_vocabulary()
embedding_dim = 16
embeddings = torch.randn(len(word_to_index), embedding_dim)

posts = [
    {"Title": "python machine learning", "score": 42},
    {"Title": "king queen man woman", "score": 7},
]
with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as handle:
    json.dump(posts, handle)
    posts_path = Path(handle.name)

title_embeddings, scores = create_title_embeddings(
    posts_path, word_to_index, embeddings, embedding_dim
)
print(f"title embeddings shape: {title_embeddings.shape}")
print(f"scores: {scores.tolist()}")

dataset = SimpleHNDataset(title_embeddings, scores)
print(
    f"dataset: {len(dataset)} examples; "
    f"first = {tuple(dataset[0][0].shape)} -> score {dataset[0][1].item():.0f}"
)

predictor = SimplePredictor(input_dim=embedding_dim)
with torch.no_grad():
    print(f"untrained predictor on the first title: {predictor(dataset[0][0]).item():.3f}")

posts_path.unlink()


## 4. Serving (`make serve`)

`serve.py` wraps the prediction path in a FastAPI app; `predict.py` holds the
functions it calls (`load_models`, `create_title_embedding`, `predict_score`). The
checkpoints are gitignored and pulled from the HF Hub at load time, so on a host that
has not trained or pulled them, `load_models()` raises `FileNotFoundError` — the
service is designed to degrade gracefully rather than crash. That is exactly what
this cell shows: the app answers `/health` before any model is loaded, and the load
fails loudly (with `hf_repo_id=None`, forcing a local-only load) instead of
pretending a model appeared. I finish by demonstrating `create_title_embedding`
against random vectors, since that step is pure and needs no checkpoint.

In [ ]:
import torch
from fastapi.testclient import TestClient

from config import Config
from predict import create_title_embedding, load_models
from serve import app
from word_embeddings.data import sample_vocabulary

# The service answers even before models are loaded. TestClient without `with` skips
# the lifespan, so /health reports the not-yet-loaded state (the test suite relies on
# this too).
client = TestClient(app)
print(f"/health -> {client.get('/health').json()}")

# Checkpoints are gitignored and pulled from the HF Hub. With hf_repo_id=None the
# load is local-only, so a host with no checkpoints on disk gets FileNotFoundError.
try:
    loaded = load_models(Config(hf_repo_id=None))
except FileNotFoundError as error:
    print("load_models() -> FileNotFoundError (expected: checkpoints are gitignored):")
    print(f"  {error}")
    print("With a real hf_repo_id this would pull them from the Hub instead.")
else:
    print(f"Local checkpoints present; loaded a {len(loaded.word_to_index)}-word vocabulary.")

# The embedding step is pure and needs no checkpoint: average the known word vectors.
word_to_index, _, _ = sample_vocabulary()
embedding_dim = 16
embeddings = torch.randn(len(word_to_index), embedding_dim)
vector = create_title_embedding(
    "python machine learning", word_to_index, embeddings, embedding_dim
)
print(
    f"create_title_embedding('python machine learning') -> "
    f"shape {tuple(vector.shape)}, norm {vector.norm().item():.3f}"
)


## Wrap-up

That is the full pipeline — corpus, embeddings, predictor, serving — with each stage
importing and calling the real modules. Stages 1 and 3 are narrated and their
offline pieces exercised rather than run end to end, because their real runs need
the network or checkpoints this environment does not have. For the numbers the full
pipeline actually produced (real corpus, real CBOW sweep, real predictor training,
and the honest finding that the predictor barely beats a mean baseline), see the
README's Experimental Results section.